# Task 2.2 — Reproduction of One Contribution (20 marks)

Reproduce the **EM algorithm with eigenvalue constraints** on a fixed two-facet PLTM and evaluate recovery of multiple latent variables using **NMI**.

**Do not clear outputs before submission.**

---

## Contribution and metric

**Contribution we reproduce:** The ability of the **EM algorithm with eigenvalue constraints** (Section 3 of the paper) to recover **multiple latent variables (facets)** when the PLTM has a **fixed** tree structure: one latent Y₁ attached to Pouch 1 (features 0, 1) and one latent Y₂ attached to Pouch 2 (features 2, 3). We do not implement full structure search; we demonstrate parameter estimation and facet recovery.

**Evaluation metric:** **Normalized Mutual Information (NMI)** between the inferred latent assignments (Z₁, Z₂) and the ground-truth facet labels, as in Section 5.2 of the paper.

## Dataset used

We use the **synthetic multifaceted dataset** from Task 2.1: 1,000 samples × 4 features, with ground-truth labels for Facet 1 (3 clusters on features 0–1) and Facet 2 (2 clusters on features 2–3). Data are loaded from `data/` (generated in task 2 1.ipynb).

In [1]:
import numpy as np
import os
from sklearn.metrics import normalized_mutual_info_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Load dataset and ground-truth facet labels from Task 2.1
DATA_DIR = "data"
X = np.load(os.path.join(DATA_DIR, "toy_multifacet_X.npy"))
labels_facet1 = np.load(os.path.join(DATA_DIR, "toy_multifacet_labels_facet1.npy"))
labels_facet2 = np.load(os.path.join(DATA_DIR, "toy_multifacet_labels_facet2.npy"))

N, D = X.shape
X1 = X[:, :2]   # Pouch 1 (features 0, 1)
X2 = X[:, 2:4]  # Pouch 2 (features 2, 3)
print("Dataset shape:", X.shape, "| Pouch 1:", X1.shape, "| Pouch 2:", X2.shape)

Dataset shape: (1000, 4) | Pouch 1: (1000, 2) | Pouch 2: (1000, 2)


The code above loads the data matrix **X** and the two ground-truth label vectors from the `data/` folder. We split **X** into **X1** (Pouch 1) and **X2** (Pouch 2) for the fixed two-pouch PLTM. This is the same dataset described in Task 2.1 and in the paper’s evaluation setup (Section 5).

## Fixed PLTM architecture

We hard-code a tree with **two independent latent variables**: Y₁ (3 states) is the parent of Pouch 1 (features 0, 1); Y₂ (2 states) is the parent of Pouch 2 (features 2, 3). There is no shared root dependency—Y₁ and Y₂ are independent—so we have two separate mixture models, one per pouch, which matches the way the toy data were generated (Section 2, Figure 1(a) of the paper).

In [2]:
# Fixed structure: latent cardinalities and eigenvalue constraint constant (Section 3, Section 5.1)
K1 = 3   # number of states for Y1 (Facet 1)
K2 = 2   # number of states for Y2 (Facet 2)
GAMMA = 20  # eigenvalue constraint: paper sets gamma = 20
MAX_ITER = 200
TOL = 1e-4   # stop when log-likelihood increase is below this

We set the number of latent states (K1=3, K2=2) to match the ground-truth facets, and γ=20 as in the paper (Section 5.1). The EM loop will run until the log-likelihood change is below `TOL` or we reach `MAX_ITER` iterations (Section 3).

In [3]:
# --- Parameter initialization (Section 3) ---
# P(Y1) and P(Y2): random probabilities in (0,1], then normalize
pi1 = np.random.uniform(0.1, 1.0, size=K1)
pi1 /= pi1.sum()
pi2 = np.random.uniform(0.1, 1.0, size=K2)
pi2 /= pi2.sum()

# Sample mean and covariance per pouch (for init and for eigenvalue bounds)
mu_D1 = X1.mean(axis=0)
cov_D1 = np.cov(X1.T)
mu_D2 = X2.mean(axis=0)
cov_D2 = np.cov(X2.T)

# Gaussian means: sample from N(mu_D, Sigma_D) per component (Section 3)
mu1 = np.array([np.random.multivariate_normal(mu_D1, cov_D1) for _ in range(K1)])
mu2 = np.array([np.random.multivariate_normal(mu_D2, cov_D2) for _ in range(K2)])

# Covariances: initialize to sample covariance of the pouch (Section 3)
cov1 = np.tile(cov_D1, (K1, 1, 1)).copy()
cov2 = np.tile(cov_D2, (K2, 1, 1)).copy()

# Ensure covariances are positive definite (add small diagonal if needed)
for k in range(K1):
    cov1[k] = cov1[k] + 1e-6 * np.eye(2)
for k in range(K2):
    cov2[k] = cov2[k] + 1e-6 * np.eye(2)
print("Initialized: pi1, pi2, mu1, mu2, cov1, cov2.")

Initialized: pi1, pi2, mu1, mu2, cov1, cov2.


We initialize the mixing proportions P(Y₁) and P(Y₂) with random values in (0, 1] then normalize. The Gaussian means for each latent state are drawn from the multivariate normal with the sample mean and covariance of that pouch; the covariances are set to the sample covariance of the pouch. This follows the initialization described in Section 3 of the paper.

In [4]:
def mvnpdf(x, mu, cov):
    """Multivariate normal log-pdf for a single vector x. Returns log density."""
    d = len(mu)
    L = np.linalg.cholesky(cov)
    y = np.linalg.solve(L, x - mu)
    log_det = 2 * np.sum(np.log(np.diag(L)))
    return -0.5 * (d * np.log(2 * np.pi) + log_det + np.dot(y, y))

Helper to compute the log-density of a multivariate Gaussian, used in the E-step to evaluate P(x|y) for each pouch and latent state (Section 2: conditional Gaussian P(x|y) = N(x | μ_y, Σ_y)).

In [5]:
# --- E-step: posterior P(Y|data) for each latent (Section 3) ---
def e_step(X1, X2, pi1, pi2, mu1, mu2, cov1, cov2):
    # Responsibilities for Y1 (Pouch 1): r1[i,k] = P(Y1=k | X1[i])
    log_r1 = np.zeros((N, K1))
    for k in range(K1):
        log_r1[:, k] = np.log(pi1[k] + 1e-300) + np.array([mvnpdf(X1[i], mu1[k], cov1[k]) for i in range(N)])
    r1 = np.exp(log_r1 - np.max(log_r1, axis=1, keepdims=True))
    r1 /= r1.sum(axis=1, keepdims=True)

    # Responsibilities for Y2 (Pouch 2): r2[i,l] = P(Y2=l | X2[i])
    log_r2 = np.zeros((N, K2))
    for l in range(K2):
        log_r2[:, l] = np.log(pi2[l] + 1e-300) + np.array([mvnpdf(X2[i], mu2[l], cov2[l]) for i in range(N)])
    r2 = np.exp(log_r2 - np.max(log_r2, axis=1, keepdims=True))
    r2 /= r2.sum(axis=1, keepdims=True)
    return r1, r2

The E-step computes the posterior probabilities P(Y₁=k | X₁) and P(Y₂=l | X₂) for every data point and every latent state. For each pouch we use the current mixing weights and Gaussian parameters; the result is the responsibility matrix (soft assignment) used in the M-step. This corresponds to the E-step in Section 3, where the algorithm computes P(y, y′ | d_i) and P(y | d_i) using inference in the mixed Bayesian network; here, with independent pouches, the posterior factorizes per pouch.

In [6]:
# --- M-step: update P(Y), means μ_y, covariances Σ_y (Section 3, Eqs. 2 & 3) ---
def m_step(X1, X2, r1, r2):
    # Update mixing proportions: P(Y=k) ∝ sum_i P(Y=k|d_i)  [Eq. 1 type]
    pi1_new = r1.sum(axis=0) / N
    pi2_new = r2.sum(axis=0) / N

    # Update means: μ_y = Σ_i P(y|d_i) x_i / Σ_i P(y|d_i)  [Eq. 2]
    mu1_new = np.zeros_like(mu1)
    for k in range(K1):
        nk = r1[:, k].sum()
        if nk > 1e-10:
            mu1_new[k] = (r1[:, k] @ X1) / nk
        else:
            mu1_new[k] = mu1[k]
    mu2_new = np.zeros_like(mu2)
    for l in range(K2):
        nl = r2[:, l].sum()
        if nl > 1e-10:
            mu2_new[l] = (r2[:, l] @ X2) / nl
        else:
            mu2_new[l] = mu2[l]

    # Update covariances: Σ_y = Σ_i P(y|d_i)(x_i - μ_y)(x_i - μ_y)' / Σ_i P(y|d_i)  [Eq. 3]
    cov1_new = np.zeros_like(cov1)
    for k in range(K1):
        nk = r1[:, k].sum()
        if nk > 1e-10:
            diff = X1 - mu1_new[k]
            cov1_new[k] = (r1[:, k] * diff.T).dot(diff) / nk
        else:
            cov1_new[k] = cov1[k]
        cov1_new[k] += 1e-8 * np.eye(2)
    cov2_new = np.zeros_like(cov2)
    for l in range(K2):
        nl = r2[:, l].sum()
        if nl > 1e-10:
            diff = X2 - mu2_new[l]
            cov2_new[l] = (r2[:, l] * diff.T).dot(diff) / nl
        else:
            cov2_new[l] = cov2[l]
        cov2_new[l] += 1e-8 * np.eye(2)
    return pi1_new, pi2_new, mu1_new, mu2_new, cov1_new, cov2_new

This block implements the M-step by updating the mixing proportions P(Y₁), P(Y₂) from the expected counts, and the means μ_y and covariance matrices Σ_y for each pouch from the posterior-weighted sample statistics. It corresponds to Equations (2) and (3) in Section 3 of the paper.

In [7]:
# --- Eigenvalue constraint (Section 3): σ²_min ≤ λ_i ≤ γ·σ²_max (γ=20) ---
def apply_eigenvalue_constraint(cov, cov_sample, gamma=GAMMA):
    """Clip eigenvalues of cov to [σ²_min, γ·σ²_max] from sample covariance diagonal."""
    sigma2_min = np.diag(cov_sample).min()
    sigma2_max = np.diag(cov_sample).max()
    lam_max = gamma * sigma2_max
    # Paper: lower bound is σ²_min (not σ²_min/γ)
    lam_min = sigma2_min
    w, v = np.linalg.eigh(cov)
    w_clipped = np.clip(w, lam_min, lam_max)
    return v @ np.diag(w_clipped) @ v.T

def constrain_covariances(cov1_new, cov2_new):
    cov1_c = np.array([apply_eigenvalue_constraint(cov1_new[k], cov_D1) for k in range(K1)])
    cov2_c = np.array([apply_eigenvalue_constraint(cov2_new[l], cov_D2) for l in range(K2)])
    return cov1_c, cov2_c

This block enforces the eigenvalue constraint on each covariance matrix: every eigenvalue is clipped to the interval **[σ²_min, γ·σ²_max]**, where σ²_min and σ²_max are the minimum and maximum of the *sample variances* (diagonal of the sample covariance) for that pouch, and γ=20. This prevents the likelihood from becoming unbounded and avoids spurious local maxima, as in Section 3 (variant of the method by Ingrassia, 2004).

In [8]:
def log_likelihood(X1, X2, pi1, pi2, mu1, mu2, cov1, cov2):
    """Total log-likelihood of the data under the two independent mixture models."""
    ll = 0.0
    for i in range(N):
        log_p1 = np.array([np.log(pi1[k] + 1e-300) + mvnpdf(X1[i], mu1[k], cov1[k]) for k in range(K1)])
        log_p2 = np.array([np.log(pi2[l] + 1e-300) + mvnpdf(X2[i], mu2[l], cov2[l]) for l in range(K2)])
        ll += np.log(np.exp(log_p1).sum() + 1e-300) + np.log(np.exp(log_p2).sum() + 1e-300)
    return ll

In [9]:
# --- EM loop (Section 3): iterate E-step, M-step, eigenvalue constraint until convergence ---
log_lik_history = [log_likelihood(X1, X2, pi1, pi2, mu1, mu2, cov1, cov2)]
for it in range(MAX_ITER):
    r1, r2 = e_step(X1, X2, pi1, pi2, mu1, mu2, cov1, cov2)
    pi1, pi2, mu1, mu2, cov1_new, cov2_new = m_step(X1, X2, r1, r2)
    cov1, cov2 = constrain_covariances(cov1_new, cov2_new)
    ll = log_likelihood(X1, X2, pi1, pi2, mu1, mu2, cov1, cov2)
    log_lik_history.append(ll)
    if ll - log_lik_history[-2] < TOL:
        print(f"EM converged at iteration {it+1}. Final log-likelihood: {ll:.4f}")
        break
else:
    print(f"EM reached max iterations ({MAX_ITER}). Final log-likelihood: {ll:.4f}")

EM converged at iteration 109. Final log-likelihood: -6709.2718


We run the EM algorithm by alternating the E-step (posterior responsibilities), M-step (update parameters), and eigenvalue constraint (clip covariances). The loop stops when the increase in log-likelihood is below `TOL` or when `MAX_ITER` is reached, as in Section 3 of the paper.

In [10]:
# Recovered latent assignments (hard): Z1 = argmax_k P(Y1=k|data), Z2 = argmax_l P(Y2=l|data)
r1_final, r2_final = e_step(X1, X2, pi1, pi2, mu1, mu2, cov1, cov2)
Z1 = r1_final.argmax(axis=1)
Z2 = r2_final.argmax(axis=1)
print("Recovered Z1 (unique):", np.unique(Z1), "| Z2 (unique):", np.unique(Z2))

Recovered Z1 (unique): [0 1 2] | Z2 (unique): [0 1]


We convert the soft responsibilities to hard cluster labels by taking the argmax for each data point. Z₁ and Z₂ are the inferred facet assignments to be compared with the ground-truth labels via NMI (Section 5.2).

In [11]:
# Evaluation: NMI between recovered latents and ground-truth facet labels (Section 5.2)
nmi_facet1 = normalized_mutual_info_score(labels_facet1, Z1)
nmi_facet2 = normalized_mutual_info_score(labels_facet2, Z2)
print("NMI(Z1, ground-truth Facet 1):", round(nmi_facet1, 4))
print("NMI(Z2, ground-truth Facet 2):", round(nmi_facet2, 4))

NMI(Z1, ground-truth Facet 1): 0.9094
NMI(Z2, ground-truth Facet 2): 0.9631


We compute the Normalized Mutual Information (NMI) between each recovered latent assignment (Z₁, Z₂) and the corresponding ground-truth facet label vector. NMI measures how well the inferred clustering agrees with the true partition; it is the same metric used in the paper (Section 5.2, Equation for NMI(C, Y)).

In [13]:
# Save recovered assignments and NMI for Task 2.3 (report and feature-curve plot)
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
np.save(os.path.join(RESULTS_DIR, "task22_Z1.npy"), Z1)
np.save(os.path.join(RESULTS_DIR, "task22_Z2.npy"), Z2)
np.savez(os.path.join(RESULTS_DIR, "task22_nmi.npz"), nmi_facet1=nmi_facet1, nmi_facet2=nmi_facet2)
np.save(os.path.join(RESULTS_DIR, "task22_log_lik_history.npy"), np.array(log_lik_history))
print("Saved Z1, Z2, NMI, log_lik_history to", RESULTS_DIR)

Saved Z1, Z2, NMI, log_lik_history to results


## Interpretation of the result

On the synthetic multifaceted dataset, the fixed two-pouch PLTM trained with EM and eigenvalue constraints achieves:

- **NMI(Z₁, Facet 1) ≈ 0.91** and  
- **NMI(Z₂, Facet 2) ≈ 0.96**.

These values are very close to 1, which means:

- The latent variable **Y₁** has learned a clustering that almost perfectly matches the **3-way ground-truth partition** in the subspace of features **0–1** (Facet 1).
- The latent variable **Y₂** has learned a clustering that almost perfectly matches the **2-way ground-truth partition** in the subspace of features **2–3** (Facet 2).

In other words, a **single model** has recovered **two distinct clusterings** on **two different subsets of variables**: one clustering along (feature 0, feature 1) and another clustering along (feature 2, feature 3). This is exactly the behaviour claimed in the paper: instead of choosing one “best” subset of variables and one clustering (as methods like CVS/LFJ would do), the PLTM-style model **facilitates variable selection** by exposing **multiple latent facets**, each tied to a subset of attributes and evaluated with the same NMI metric used in the paper (Section 5.2).